In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

class AIS_Competitive:
    def __init__(self, num_detectors=100, generations=30, mutation_rate=0.1, bias=1.05, random_seed=42):
        self.num_detectors = num_detectors
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.bias = bias
        self.detectors = [] 
        self.scaler = StandardScaler() 
        self.pca = PCA(n_components=50) 
        self.rng = np.random.default_rng(random_seed)
        self.normal_memory = [] # We keep a reference to Normal data for the competition

    def preprocess(self, X, train=True):
        if train:
            X_pca = self.pca.fit_transform(X)
            return self.scaler.fit_transform(X_pca)
        else:
            X_pca = self.pca.transform(X)
            return self.scaler.transform(X_pca)

    def fit(self, normal_data, disease_data):
        print(f"Evolution Phase: Training {self.num_detectors} Competitive Detectors")
        self.normal_memory = normal_data 
        feature_dim = normal_data.shape[1]
        
        # 1. Initialize Detectors
        seed_indices = self.rng.integers(0, len(disease_data), size=self.num_detectors)
        population = disease_data[seed_indices].copy()
        
        for gen in range(self.generations):
            # 2. Mutate
            noise = self.rng.normal(0, self.mutation_rate, population.shape)
            mutants = population + noise
            
            # 3. Evaluate Fitness
            scores = []
            
            # Calculate similarity of Disease Samples to Mutants
            # shape: [n_disease_samples x n_mutants]
            sim_to_mutants = cosine_similarity(disease_data, mutants)
            
            # Calculate similarity of Disease Samples to Known Normals
            # shape: [n_disease_samples x n_normals]
            sim_to_normals = cosine_similarity(disease_data, self.normal_memory)
            
            max_sim_normal = sim_to_normals.max(axis=1)
            
            # A mutant gets a point if it is CLOSER (higher sim) to a disease sample 
            for i in range(self.num_detectors):
                margins = sim_to_mutants[:, i] - max_sim_normal
                
                score = np.sum(margins[margins > 0])
                scores.append(score)
            
            scores = np.array(scores)
            
            # 4. Selection 
            best_indices = np.argsort(scores)[::-1]
            
            # Update population with the best mutants found so far
            population = mutants[best_indices]
            
            if gen % 10 == 0:
                print(f"   Gen {gen}: Best Affinity Score = {scores.max():.4f}")
                
        self.detectors = population
        print("Evolution Complete.")

    def predict(self, samples):
        # COMPETITIVE LOGIC:
        # A sample is "Disease" if:
        # Similarity(Sample, Nearest Detector) > Similarity(Sample, Nearest Normal Training Sample) * Bias
        
        # 1. Similarity to Detectors
        sim_to_detectors = cosine_similarity(samples, self.detectors)
        max_sim_detectors = sim_to_detectors.max(axis=1)
        
        # 2. Similarity to Known Normals (The "Self" Database)
        sim_to_normals = cosine_similarity(samples, self.normal_memory)
        max_sim_normals = sim_to_normals.max(axis=1)
        
        # 3. Competition
        # Bias factor: 
        # > 1.0 : Harder to flag as disease (Reduces False Positives)
        # < 1.0 : Easier to flag as disease (Reduces False Negatives)
        bias = self.bias #1.05 
        
        predictions = (max_sim_detectors > (max_sim_normals * bias)).astype(int)
        
        winning_detectors_indices = np.where(np.any(sim_to_detectors > (max_sim_normals[:, None] * bias), axis=0))[0]
        
        return predictions, len(winning_detectors_indices)


df = pd.read_csv('GSE33000_Top10000_Var.csv', index_col=0)
labels = df['Diagnosis']
data = df.drop('Diagnosis', axis=1).values

normal_indices = np.where(labels.str.contains("C"))[0]
disease_indices = np.where(~labels.str.contains("C"))[0]

# Split (70/30)
n_split = int(len(normal_indices) * 0.7)
d_split = int(len(disease_indices) * 0.7)

train_normal_idx = normal_indices[:n_split]
test_normal_idx = normal_indices[n_split:]

train_disease_idx = disease_indices[:d_split]
test_disease_idx = disease_indices[d_split:]

X_train_normal = data[train_normal_idx]
X_train_disease = data[train_disease_idx]
X_test = data[np.concatenate([test_normal_idx, test_disease_idx])]
y_test = np.array([0]*len(test_normal_idx) + [1]*len(test_disease_idx))

print(f"Training: {len(X_train_normal)} Normal, {len(X_train_disease)} Disease")

# Run AIS
ais = AIS_Competitive(num_detectors=1000, generations=100, mutation_rate=0.2, bias=1.05)

# Preprocess
X_combined_train = np.vstack((X_train_normal, X_train_disease))
ais.preprocess(X_combined_train, train=True)

X_norm_scaled = ais.preprocess(X_train_normal, train=False)
X_dis_scaled = ais.preprocess(X_train_disease, train=False)

# Fit
ais.fit(X_norm_scaled, X_dis_scaled)

# Predict
X_test_scaled = ais.preprocess(X_test, train=False)
y_pred, num_active = ais.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nActive Memory Cells: {num_active}")
print(f"\nAccuracy: {acc:.2%}")
print("\nConfusion Matrix:")
print(f"True Normal:      {cm[0][0]}")
print(f"False Positive:   {cm[0][1]} (Adjust 'bias' UP to fix this)")
print(f"False Negative:   {cm[1][0]} (Adjust 'bias' DOWN to fix this)")
print(f"True Positive:    {cm[1][1]}")